# ERP003950 (мышь) — dedup → empirical ISS error model → PCR#1 → fragmentation → PCR#2 → 2×150

Единственный актуальный симуляционный ноутбук. Две отдельные стадии амплификации,
как в реальном 5'RACE протоколе: RT-PCR полноразмерных молекул **до** фрагментации,
и library-prep PCR отдельных фрагментов **после** фрагментации.

**Порядок:**

1. Берём `AssemblePairs.py` merged reads каждого sample.
2. **Exact-sequence deduplication** внутри sample: одинаковые merged последовательности схлопываются в один template.
3. Строим empirical InSilicoSeq 150-bp error model (обучена на РЕАЛЬНЫХ ридах, обрезанных до 150bp).
4. **PCR#1** (branching, на полноразмерных templates): `N_next = N + Binomial(N, efficiency)`.
5. **Фрагментация**: для каждого template явно нарезаем несколько кандидат-фрагментов
   (случайная длина/старт) — фрагмент существует как отслеживаемая сущность
   (`{template_id, start, length, sequence}`), а не эфемерно внутри `iss generate`.
6. **PCR#2** (library-prep, branching на уровне отдельных фрагментов, свои циклы/efficiency).
7. Финальный read budget аллоцируется мультиномиально по пулу **фрагментов** (не templates).
8. `iss generate --sequence_type amplicon` сиквенирует каждый уже нарезанный фрагмент
   целиком с двух концов — без повторной внутренней нарезки.

> **Ограничение без UMI:** exact-sequence dedup не восстанавливает истинное число молекул до PCR#1.
> Поэтому default `STARTING_COPIES_MODE="one_per_unique"` — консервативный и воспроизводимый выбор.


## Единственная директория хранения

- Output directory: `results/ERP003950/simulated/insilicoseq/`
- Template name: `{sample}_templates.fasta` (собственный dedup, без внешних зависимостей)
- С `FORCE=True` и `CLEAN_OLD_OUTPUTS=True` предыдущие templates, read-count файлы,
  ISS-модель, QC, логи и simulated FASTQ под этим же деревом удаляются перед пересборкой.
- Raw/merged входные данные вне `OUT_BASE` этот ноутбук не трогает.


## 1. Environment

In [ ]:
import os, sys, sysconfig, subprocess, time, gzip, csv, math
from pathlib import Path
import shutil
from collections import Counter
import numpy as np

_ENV_CANDIDATES = [
    "/data/user/epishkin/conda/envs/bcr_env",
    "/opt/conda/envs/bcr_env",
]
_CONDA_ENV = next((p for p in _ENV_CANDIDATES if os.path.isdir(p + "/bin")), _ENV_CANDIDATES[-1])
os.environ["PATH"] = _CONDA_ENV + "/bin:" + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
sys.path[:] = [p for p in sys.path if "/data/user/epishkin/.local" not in p]
for _site in [
    _CONDA_ENV + "/lib/python3.11/site-packages",
    _CONDA_ENV + "/lib/python3.12/site-packages",
    sysconfig.get_path("purelib"),
]:
    if os.path.isdir(_site) and _site not in sys.path:
        sys.path.insert(0, _site)

for tool in ("iss", "bowtie2", "bowtie2-build", "samtools"):
    p = subprocess.run(["which", tool], capture_output=True, text=True).stdout.strip()
    if not p:
        raise RuntimeError(f"Required tool not found: {tool}")
    print(f"{tool}: {p}")
print("numpy:", np.__version__)

## 2. Configuration

In [ ]:
VOLUME = Path("/data/user/epishkin")
DATASET = "ERP003950"
SAMPLES = ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]

MERGED_FASTQ_DIR = VOLUME / "results" / DATASET / "merged" / "fastq"
RAW_FASTQ_DIR = VOLUME / "raw" / DATASET
TARGET_READ_LENGTH = 150

OUT_BASE = VOLUME / "results" / DATASET / "simulated" / "insilicoseq"  # single canonical simulation output dir
TEMPLATES_DIR = OUT_BASE / "templates"
MODEL_DIR = OUT_BASE / "model"
COUNTS_DIR = OUT_BASE / "read_counts"
FASTQ_DIR = OUT_BASE / "fastq"
LOGS_DIR = OUT_BASE / "logs"
QC_DIR = OUT_BASE / "qc"
for d in (TEMPLATES_DIR, MODEL_DIR, COUNTS_DIR, FASTQ_DIR, LOGS_DIR, QC_DIR):
    d.mkdir(parents=True, exist_ok=True)

NPROC = 8
SEED = 42
FORCE = False
CLEAN_OLD_OUTPUTS = False  # only takes effect together with FORCE=True
COMPRESS = True

# Dedup / template policy
MIN_TEMPLATE_LENGTH = 180
DROP_SEQUENCES_WITH_N = True
STARTING_COPIES_MODE = "one_per_unique"  # recommended without UMI
# Alternative only for sensitivity analysis: "observed_multiplicity"

# PCR model
PCR_CYCLES = 25
PCR_EFFICIENCY_MEAN = 0.85       # per-molecule copying probability / cycle
PCR_EFFICIENCY_CONCENTRATION = 80.0  # larger => less template-to-template efficiency variation
PCR_MAX_COPIES = 10**12          # safety cap; far above what matters for relative sampling

# Fragmentation: candidate fragment windows are drawn explicitly below (not by ISS)
SEQUENCE_TYPE = "amplicon"  # fragments are pre-cut; iss generate must NOT re-fragment them
FRAGMENT_LENGTH_MEAN = 200
FRAGMENT_LENGTH_SD = 40
MIN_FRAGMENT_LENGTH = TARGET_READ_LENGTH + 10  # amplicon mode needs len(fragment) > read_length
FRAGMENTS_PER_TEMPLATE = 5  # candidate fragment windows sampled per template
MIN_READ_PAIRS_PER_TEMPLATE = 0  # IMPORTANT: allow sequencing dropout

# Library-prep PCR (round 2, AFTER fragmentation, on individual fragments)
LIBRARY_PCR_CYCLES = 12
LIBRARY_PCR_EFFICIENCY_MEAN = 0.85
LIBRARY_PCR_EFFICIENCY_CONCENTRATION = 80.0
LIBRARY_PCR_MAX_COPIES = 10**9

# Sequencing depth. Default: same number of PE pairs as the real raw sample.
READ_BUDGET_MODE = "match_raw_pairs"  # or "fixed"
FIXED_READ_PAIRS = 500_000

REF_FASTA = MODEL_DIR / f"{DATASET}_dedup_pseudoref.fasta"
BOWTIE2_INDEX = MODEL_DIR / f"{DATASET}_dedup_bt2"
BAM_PATH = MODEL_DIR / f"{DATASET}_raw_{TARGET_READ_LENGTH}bp_vs_dedup.bam"
CUSTOM_MODEL_PREFIX = MODEL_DIR / f"{DATASET}_empirical_{TARGET_READ_LENGTH}bp"
CUSTOM_MODEL_NPZ = Path(str(CUSTOM_MODEL_PREFIX) + ".npz")

print("OUT_BASE:", OUT_BASE)

In [ ]:
# ---------------------------------------------------------------------
# Clean generated artifacts from previous runs
# ---------------------------------------------------------------------
# Canonical output tree:
#   results/ERP003950/simulated/insilicoseq/
#
# With FORCE=True and CLEAN_OLD_OUTPUTS=True, outputs produced by the
# previous simulation run are removed and rebuilt. Source/raw data
# outside OUT_BASE are never touched.

def clean_previous_generated_outputs():
    if not (FORCE and CLEAN_OLD_OUTPUTS):
        print("Cleanup disabled.")
        return

    generated_dirs = [
        TEMPLATES_DIR,
        MODEL_DIR,
        COUNTS_DIR,
        FASTQ_DIR,
        QC_DIR,
        LOGS_DIR,
    ]

    for directory in generated_dirs:
        directory = Path(directory)
        if not directory.exists():
            continue
        for path in directory.iterdir():
            if path.is_file() or path.is_symlink():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)

    for directory in generated_dirs:
        Path(directory).mkdir(parents=True, exist_ok=True)

    print(f"Cleaned previous generated outputs under: {OUT_BASE}")

clean_previous_generated_outputs()


## 3. Helpers

In [ ]:
def open_text(path, mode="rt"):
    return gzip.open(path, mode) if str(path).endswith(".gz") else open(path, mode)

def iter_fastq(path):
    with open_text(path, "rt") as h:
        while True:
            head = h.readline()
            if not head:
                return
            seq = h.readline().rstrip("\n\r")
            plus = h.readline()
            qual = h.readline()
            if not plus or not qual:
                raise ValueError(f"Truncated FASTQ: {path}")
            yield seq

def count_fastq(path):
    return sum(1 for _ in iter_fastq(path))

def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

def run_with_heartbeat(cmd, log_path, heartbeat=30, shell=False):
    t0 = time.time()
    with open(log_path, "w") as log_h:
        proc = subprocess.Popen(cmd, stdout=log_h, stderr=subprocess.STDOUT, text=True, shell=shell)
        label = cmd if shell else " ".join(map(str, cmd))
        print(f"[run] {label}\n  pid={proc.pid} log={log_path}")
        while proc.poll() is None:
            print(f"  running: elapsed={(time.time()-t0)/60:.1f} min", flush=True)
            time.sleep(heartbeat)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}); see {log_path}")
    print(f"done in {(time.time()-t0)/60:.1f} min")

def sample_seed(sample, extra=0):
    return int(SEED + extra + sum((i + 1) * ord(ch) for i, ch in enumerate(sample)))

## 4. Exact-sequence deduplication of merged reads

Дедуп выполняется **внутри каждого sample** по полной merged nucleotide sequence после `upper()`.

- Все полностью одинаковые merged reads → один `template_id`.
- `observed_multiplicity` = сколько merged reads было схлопнуто.
- Templates с `N` (если включено) и слишком короткие sequences удаляются.
- Для simulation default `starting_copies=1`; multiplicity не трактуется как pre-PCR molecule count.

In [ ]:
def deduplicate_sample(sample, force=FORCE):
    in_fq = MERGED_FASTQ_DIR / f"{sample}_assemble-pass.fastq.gz"
    out_fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    out_tsv = TEMPLATES_DIR / f"{sample}_template_qc.tsv"

    if out_fa.exists() and out_tsv.exists() and not force:
        print(f"[{sample}] [skip] dedup outputs exist")
        return
    if not in_fq.exists():
        raise FileNotFoundError(in_fq)

    counts = Counter()
    raw_n = kept_n = dropped_short = dropped_n = 0
    for seq in iter_fastq(in_fq):
        raw_n += 1
        seq = seq.upper()
        if len(seq) < MIN_TEMPLATE_LENGTH:
            dropped_short += 1
            continue
        if DROP_SEQUENCES_WITH_N and "N" in seq:
            dropped_n += 1
            continue
        counts[seq] += 1
        kept_n += 1

    # Stable deterministic order: abundance desc, then sequence lexicographically.
    ordered = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
    with open(out_fa, "w") as fa, open(out_tsv, "w", newline="") as ts:
        w = csv.writer(ts, delimiter="\t")
        w.writerow(["template_id", "observed_multiplicity", "length"])
        for i, (seq, mult) in enumerate(ordered, 1):
            tid = f"{sample}_tpl_{i:07d}"
            fa.write(f">{tid}\n{seq}\n")
            w.writerow([tid, mult, len(seq)])

    unique_n = len(ordered)
    dup_fraction = 1 - unique_n / kept_n if kept_n else float("nan")
    print(f"[{sample}] merged={raw_n:,} kept={kept_n:,} unique={unique_n:,} "
          f"exact-duplicate fraction={dup_fraction:.3f} dropped_short={dropped_short:,} dropped_N={dropped_n:,}")

for sample in SAMPLES:
    deduplicate_sample(sample)

## 5. Dedup QC summary

In [ ]:
dedup_summary = []
for sample in SAMPLES:
    q = TEMPLATES_DIR / f"{sample}_template_qc.tsv"
    rows = []
    with open(q) as h:
        r = csv.DictReader(h, delimiter="\t")
        for x in r:
            rows.append(x)
    multiplicities = np.array([int(x["observed_multiplicity"]) for x in rows], dtype=np.int64)
    dedup_summary.append({
        "sample": sample,
        "unique_templates": len(rows),
        "merged_reads_represented": int(multiplicities.sum()),
        "singletons": int((multiplicities == 1).sum()),
        "max_observed_multiplicity": int(multiplicities.max()) if len(multiplicities) else 0,
    })

summary_path = QC_DIR / "dedup_summary.tsv"
with open(summary_path, "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=dedup_summary[0].keys(), delimiter="\t")
    w.writeheader(); w.writerows(dedup_summary)
for r in dedup_summary:
    print(r)
print("wrote", summary_path)

## 6. Build empirical 150-bp InSilicoSeq error model

Здесь deduplicated merged sequences используются как **pseudo-reference**, а реальные raw R1/R2 — как наблюдаемые reads.

Это лучше, чем индексировать миллионы duplicate merged records, но это всё равно empirical pseudo-reference model: merged sequence сама получена из sequencing reads, а не из независимого ground truth. Для BCR без UMI/known truth это практический компромисс; модель sequencing errors и модель PCR остаются раздельными.

In [ ]:
# 6a. Concatenate deduplicated templates into one pseudo-reference.
if REF_FASTA.exists() and not FORCE:
    print("[skip]", REF_FASTA)
else:
    n = 0
    with open(REF_FASTA, "w") as out:
        for sample in SAMPLES:
            fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
            for tid, seq in iter_fasta(fa):
                out.write(f">{tid}\n{seq}\n")
                n += 1
    print(f"wrote {REF_FASTA}: {n:,} unique templates")

# 6b. bowtie2-build
marker = Path(str(BOWTIE2_INDEX) + ".1.bt2")
marker_large = Path(str(BOWTIE2_INDEX) + ".1.bt2l")
if (marker.exists() or marker_large.exists()) and not FORCE:
    print("[skip] bowtie2 index exists")
else:
    run_with_heartbeat(
        ["bowtie2-build", "--threads", str(NPROC), str(REF_FASTA), str(BOWTIE2_INDEX)],
        LOGS_DIR / "bowtie2_build.log",
    )

In [ ]:
# 6c. Align the original full-length real PE reads against

if BAM_PATH.exists() and Path(str(BAM_PATH) + ".bai").exists() and not FORCE:
    print("[skip]", BAM_PATH)

else:
    r1_list = ",".join(
        str(RAW_FASTQ_DIR / f"{s}_1.fastq.gz")
        for s in SAMPLES)
    r2_list = ",".join(
        str(RAW_FASTQ_DIR / f"{s}_2.fastq.gz")
        for s in SAMPLES)

    cmd = (
        f"bowtie2 --local --no-mixed --no-discordant --trim-to {TARGET_READ_LENGTH} "
        f"-p {NPROC} "
        f"-x {BOWTIE2_INDEX} "
        f"-1 {r1_list} "
        f"-2 {r2_list} "
        f"2> {LOGS_DIR / 'bowtie2_align.stderr.log'} "
        f"| samtools view -b - "
        f"| samtools sort -@ {NPROC} -o {BAM_PATH} -")

    run_with_heartbeat(
        cmd,
        LOGS_DIR / "bowtie2_align_pipe.log",
        shell=True,)

    subprocess.run(
        ["samtools", "index", str(BAM_PATH)],
        check=True,)

    print("indexed", BAM_PATH)

# Basic alignment QC
stats = subprocess.run(
    ["samtools", "flagstat", str(BAM_PATH)],
    capture_output=True,
    text=True,
    check=True,
).stdout

print(stats[:2500])

In [ ]:
# 6d. Fit ISS KDE error model and validate output/read length.
if CUSTOM_MODEL_NPZ.exists() and not FORCE:
    print("[skip] model exists:", CUSTOM_MODEL_NPZ)
else:
    run_with_heartbeat(
        ["iss", "model", "--debug", "-b", str(BAM_PATH), "-o", str(CUSTOM_MODEL_PREFIX)],
        LOGS_DIR / "iss_model.log",
    )
    if not CUSTOM_MODEL_NPZ.exists():
        tail = (LOGS_DIR / "iss_model.log").read_text()[-4000:]
        raise RuntimeError(f"iss model did not create {CUSTOM_MODEL_NPZ}. Log tail:\n{tail}")

from iss.error_models.kde import KDErrorModel
em = KDErrorModel(str(CUSTOM_MODEL_NPZ))
print("model:", CUSTOM_MODEL_NPZ)
print("model read_length:", em.read_length)
assert em.read_length == TARGET_READ_LENGTH, (em.read_length, TARGET_READ_LENGTH)

## 7. Sequencing budget per sample

По умолчанию выходная глубина равна числу read pairs в реальном raw R1 FASTQ соответствующего sample. Это **отдельный** параметр от PCR: PCR меняет состав амплифицированного пула, sequencing budget определяет только сколько пар мы из него считываем.

In [ ]:
def get_read_budget(sample):
    if READ_BUDGET_MODE == "fixed":
        return int(FIXED_READ_PAIRS)
    if READ_BUDGET_MODE == "match_raw_pairs":
        r1 = RAW_FASTQ_DIR / f"{sample}_1.fastq.gz"
        r2 = RAW_FASTQ_DIR / f"{sample}_2.fastq.gz"
        n1 = count_fastq(r1)
        n2 = count_fastq(r2)
        if n1 != n2:
            raise ValueError(f"{sample}: R1/R2 count mismatch: {n1} vs {n2}")
        return n1
    raise ValueError(f"Unknown READ_BUDGET_MODE={READ_BUDGET_MODE!r}")

READ_BUDGETS = {s: get_read_budget(s) for s in SAMPLES}
print(READ_BUDGETS)

## 8. PCR#1 — branching PCR on whole templates (pre-fragmentation)

Для каждого template и каждого цикла:

\[
N_{c+1}=N_c+\mathrm{Binomial}(N_c,p_i)
\]

где `p_i` — template-specific efficiency. Она один раз берётся из Beta distribution
с заданным mean/concentration. Поэтому модель включает:

- stochastic early-cycle jackpotting;
- template-to-template amplification efficiency variation;
- естественный PCR dropout/under-amplification;
- отсутствие искусственного требования «минимум 1 simulated read на template».

PCR-copy counts могут быть огромны, но мы не материализуем молекулы: храним только integer abundance.

Это RT-PCR полноразмерных 5'RACE-молекул — **до** фрагментации.


In [ ]:
def load_dedup_templates(sample):
    fa = TEMPLATES_DIR / f"{sample}_templates.fasta"
    qc = TEMPLATES_DIR / f"{sample}_template_qc.tsv"
    mult = {}
    with open(qc) as h:
        for r in csv.DictReader(h, delimiter="\t"):
            mult[r["template_id"]] = int(r["observed_multiplicity"])
    rows = []
    for tid, seq in iter_fasta(fa):
        rows.append({"template_id": tid, "sequence": seq, "length": len(seq),
                     "observed_multiplicity": mult[tid]})
    return rows

def starting_copies(row):
    if STARTING_COPIES_MODE == "one_per_unique":
        return 1
    if STARTING_COPIES_MODE == "observed_multiplicity":
        return row["observed_multiplicity"]
    raise ValueError(STARTING_COPIES_MODE)

def branching_pcr(n0, efficiency, cycles, rng):
    n = int(n0)
    for _ in range(cycles):
        if n <= 0:
            return 0
        new = int(rng.binomial(n, efficiency))
        n += new
        if n >= PCR_MAX_COPIES:
            return int(PCR_MAX_COPIES)
    return n

def simulate_pcr_pool(sample):
    rng = np.random.default_rng(sample_seed(sample, 1000))
    rows = load_dedup_templates(sample)

    mean = PCR_EFFICIENCY_MEAN
    conc = PCR_EFFICIENCY_CONCENTRATION
    if not (0 < mean <= 1):
        raise ValueError("PCR_EFFICIENCY_MEAN must be in (0,1]")
    if mean == 1:
        alpha = beta = None
    else:
        alpha, beta = mean * conc, (1 - mean) * conc

    for r in rows:
        p = 1.0 if mean == 1 else float(rng.beta(alpha, beta))
        n0 = starting_copies(r)
        n_pcr = branching_pcr(n0, p, PCR_CYCLES, rng)
        r.update(starting_copies=n0, pcr_efficiency=p, pcr_copies=n_pcr)
    return rows

# Smoke test: deterministic with fixed seed and no negative values.
_test_rng = np.random.default_rng(1)
assert branching_pcr(1, 1.0, 10, _test_rng) == 2**10
assert branching_pcr(1, 0.0, 10, _test_rng) == 1
print("branching PCR smoke tests: OK")

## 8b. Фрагментация — явный, отслеживаемый пул фрагментов

В отличие от `iss generate --sequence_type metagenomics` (где фрагмент рождается заново
внутри каждого вызова и нигде не сохраняется), здесь фрагменты материализуются как
обычные Python-объекты `{fragment_id, template_id, start, length, sequence}` **до**
второй ПЦР — иначе фрагмент нечего было бы отдельно амплифицировать.

Для каждого template из PCR#1 (`pcr_copies > 0`) тянем `FRAGMENTS_PER_TEMPLATE`
кандидат-окон: длина ~ Normal(`FRAGMENT_LENGTH_MEAN`, `FRAGMENT_LENGTH_SD`), обрезана
снизу `MIN_FRAGMENT_LENGTH` (иначе `iss generate --sequence_type amplicon` не сможет
вырезать риды нужной длины), случайный старт внутри template под эту длину.

Начальное количество копий фрагмента = `pcr_copies` родительского template (упрощение:
каждый фрагментный "вид" стартует с той же abundance, что и весь пул амплифицированных
молекул этого template — а не делит его между собой).


In [ ]:
def enumerate_fragments(sample, rows_pcr1, rng):
    fragments = []
    for r in rows_pcr1:
        if r["pcr_copies"] <= 0:
            continue
        seq = r["sequence"]
        tlen = r["length"]
        max_len = min(tlen, int(FRAGMENT_LENGTH_MEAN + 4 * FRAGMENT_LENGTH_SD))
        if max_len < MIN_FRAGMENT_LENGTH:
            continue  # template too short to yield a usable fragment
        for k in range(FRAGMENTS_PER_TEMPLATE):
            flen = int(rng.normal(FRAGMENT_LENGTH_MEAN, FRAGMENT_LENGTH_SD))
            flen = max(MIN_FRAGMENT_LENGTH, min(flen, tlen))
            start = int(rng.integers(0, tlen - flen + 1))
            fragments.append({
                "fragment_id": f"{r['template_id']}_frag{k}",
                "template_id": r["template_id"],
                "template_length": tlen,
                "fragment_start": start,
                "fragment_length": flen,
                "sequence": seq[start:start + flen],
                "template_pcr1_copies": r["pcr_copies"],
            })
    return fragments


## 8c. PCR#2 — library-prep PCR на отдельных фрагментах (после фрагментации)

Та же branching-модель, что и PCR#1, но применяется **независимо к каждому фрагменту**
(не к template целиком) — это и есть вторая амплификация: часть фрагментов одного и того
же template может уйти в разы вперёд по числу копий, а часть — отстать, за счёт своей
собственной стохастической efficiency в library PCR. Обычно меньше циклов, чем у RT-PCR
(`LIBRARY_PCR_CYCLES=12` против `PCR_CYCLES=25`), т.к. library/index PCR на практике короче.


In [ ]:
def branching_pcr_vectorized(n0, efficiency, cycles, max_copies, rng):
    n = np.asarray(n0, dtype=np.int64).copy()
    eff = np.asarray(efficiency, dtype=np.float64)
    for _ in range(cycles):
        active = n > 0
        if not active.any():
            break
        new = np.zeros_like(n)
        new[active] = rng.binomial(n[active], eff[active])
        n = n + new
        np.minimum(n, max_copies, out=n)
    return n


def run_pcr2_on_fragments(fragments, rng):
    mean = LIBRARY_PCR_EFFICIENCY_MEAN
    conc = LIBRARY_PCR_EFFICIENCY_CONCENTRATION
    if mean == 1:
        efficiency = np.ones(len(fragments))
    else:
        alpha, beta = mean * conc, (1 - mean) * conc
        efficiency = rng.beta(alpha, beta, size=len(fragments))
    n0 = np.array([f["template_pcr1_copies"] for f in fragments], dtype=np.int64)
    copies = branching_pcr_vectorized(n0, efficiency, LIBRARY_PCR_CYCLES, LIBRARY_PCR_MAX_COPIES, rng)
    for f, eff, cp in zip(fragments, efficiency, copies):
        f["library_pcr_efficiency"] = float(eff)
        f["library_pcr2_copies"] = int(cp)
    return fragments

# Smoke test: deterministic with fixed seed, matches scalar branching_pcr for a single row.
_test_rng = np.random.default_rng(1)
_vec = branching_pcr_vectorized(np.array([1]), np.array([1.0]), 10, 10**12, _test_rng)
assert int(_vec[0]) == 2**10
print("vectorized branching PCR smoke test: OK")


## 9. Allocate exact sequencing budget across fragments

Sampling weight for fragment `j` is просто его abundance после PCR#2:

\[
w_j = N_j^{PCR2}
\]

(веса по "valid fragment starts" больше не нужны — длина/позиция каждого фрагмента уже
зафиксирована на шаге 8b, а не оценивается статистически.)

Один мультиномиальный draw аллоцирует ровно `target_read_pairs` по всем фрагментам всех
templates сэмпла разом. Фрагменты (и целые templates) могут получить ноль ридов —
это ожидаемый sampling dropout.


In [ ]:
def allocate_reads_over_fragments(sample, force=FORCE):
    out_counts = COUNTS_DIR / f"{sample}_read_counts.tsv"
    out_frags_fa = TEMPLATES_DIR / f"{sample}_fragments.fasta"
    out_qc = QC_DIR / f"{sample}_pcr_allocation.tsv"
    out_pcr1_qc = QC_DIR / f"{sample}_pcr1_template_qc.tsv"
    if out_counts.exists() and out_frags_fa.exists() and out_qc.exists() and not force:
        print(f"[{sample}] [skip] fragment allocation exists")
        return

    rows_pcr1 = simulate_pcr_pool(sample)

    fields1 = ["template_id", "length", "observed_multiplicity", "starting_copies",
               "pcr_efficiency", "pcr_copies"]
    with open(out_pcr1_qc, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields1, delimiter="\t")
        w.writeheader(); w.writerows({k: r[k] for k in fields1} for r in rows_pcr1)

    rng = np.random.default_rng(sample_seed(sample, 2000))
    fragments = enumerate_fragments(sample, rows_pcr1, rng)
    fragments = run_pcr2_on_fragments(fragments, rng)

    weights = np.asarray([f["library_pcr2_copies"] for f in fragments], dtype=np.float64)
    total = weights.sum()
    if not np.isfinite(total) or total <= 0:
        raise RuntimeError(f"{sample}: invalid total sampling weight {total}")
    probs = weights / total
    target = int(READ_BUDGETS[sample])
    allocated = rng.multinomial(target, probs)

    with open(out_counts, "w", newline="") as h, open(out_frags_fa, "w") as fa_h:
        counts_w = csv.writer(h, delimiter="\t")
        for f, n in zip(fragments, allocated):
            f["simulated_read_pairs"] = int(n)
            if n > 0:
                # iss generate --readcount_file sums to TOTAL reads (R1+R2), then
                # does n_reads // 2 internally -- write 2*pairs so the actual
                # generated pair count matches our intended read budget.
                counts_w.writerow([f["fragment_id"], int(n) * 2])
                fa_h.write(f">{f['fragment_id']}\n{f['sequence']}\n")

    fields2 = ["fragment_id", "template_id", "template_length", "fragment_start",
               "fragment_length", "template_pcr1_copies", "library_pcr_efficiency",
               "library_pcr2_copies", "simulated_read_pairs"]
    with open(out_qc, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fields2, delimiter="\t")
        w.writeheader(); w.writerows({k: f[k] for k in fields2} for f in fragments)

    nonzero = int((allocated > 0).sum())
    templates_represented = len({f["template_id"] for f, n in zip(fragments, allocated) if n > 0})
    print(f"[{sample}] target={target:,} allocated={allocated.sum():,} "
          f"fragments={len(fragments):,} fragments_with_reads={nonzero:,} "
          f"templates_represented={templates_represented:,} "
          f"fragment_dropout={1 - nonzero/len(fragments):.3f}")
    assert int(allocated.sum()) == target

for sample in SAMPLES:
    allocate_reads_over_fragments(sample)


## 10. PCR/allocation QC summary

In [ ]:
pcr_summary = []
for sample in SAMPLES:
    q = QC_DIR / f"{sample}_pcr_allocation.tsv"
    copies2 = []
    reads = []
    eff2 = []
    templates_seen = set()
    with open(q) as h:
        for r in csv.DictReader(h, delimiter="\t"):
            copies2.append(int(r["library_pcr2_copies"]))
            reads.append(int(r["simulated_read_pairs"]))
            eff2.append(float(r["library_pcr_efficiency"]))
            templates_seen.add(r["template_id"])
    copies2 = np.asarray(copies2); reads = np.asarray(reads); eff2 = np.asarray(eff2)
    pcr_summary.append({
        "sample": sample,
        "fragments": len(reads),
        "templates_with_fragments": len(templates_seen),
        "target_read_pairs": int(reads.sum()),
        "fragments_with_reads": int((reads > 0).sum()),
        "fragment_sampling_dropout": float((reads == 0).mean()),
        "mean_library_pcr_efficiency": float(eff2.mean()),
        "median_library_pcr2_copies": float(np.median(copies2)),
        "max_library_pcr2_copies": int(copies2.max()),
    })

out = QC_DIR / "pcr_summary.tsv"
with open(out, "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=pcr_summary[0].keys(), delimiter="\t")
    w.writeheader(); w.writerows(pcr_summary)
for r in pcr_summary:
    print(r)
print("wrote", out)


## 11. Generate 2×150 reads with InSilicoSeq (`amplicon` mode)

`--genomes` теперь указывает на `{sample}_fragments.fasta` (материализованные на шаге 9
фрагменты, а не полноразмерные templates). `--sequence_type amplicon` секвенирует каждую
запись целиком с обоих концов, **без** внутренней повторной нарезки (см. `iss/generator.py:
simulate_read()` — при `amplicon` `forward_start=0`, `reverse_start=len(seq)-read_length`).
Поэтому `--fragment-length`/`--fragment-length-sd` здесь не передаются — они относились бы
к нарезке, которая уже явно сделана на шаге 8b.


In [ ]:
def run_iss_generate(sample, force=FORCE):
    fragments_fa = TEMPLATES_DIR / f"{sample}_fragments.fasta"
    counts_tsv = COUNTS_DIR / f"{sample}_read_counts.tsv"
    out_prefix = FASTQ_DIR / sample
    ext = ".fastq.gz" if COMPRESS else ".fastq"
    r1_out = Path(str(out_prefix) + f"_R1{ext}")
    r2_out = Path(str(out_prefix) + f"_R2{ext}")

    if r1_out.exists() and r2_out.exists() and not force:
        print(f"[{sample}] [skip] FASTQ exists")
        return

    if not fragments_fa.exists():
        raise FileNotFoundError(f"Missing fragments FASTA for {sample}: {fragments_fa}\n"
                                 "Run step 9 (allocate_reads_over_fragments) first.")

    cmd = [
        "iss", "generate",
        "--genomes", str(fragments_fa),
        "--readcount_file", str(counts_tsv),
        "--sequence_type", SEQUENCE_TYPE,  # "amplicon" -- fragments are pre-cut
        "--model", str(CUSTOM_MODEL_NPZ),
        "--cpus", str(NPROC),
        "--output", str(out_prefix),
        "--seed", str(sample_seed(sample, 3000)),
    ]
    if COMPRESS:
        cmd.append("--compress")

    run_with_heartbeat(cmd, LOGS_DIR / f"{sample}_iss_generate.log")
    n1, n2 = count_fastq(r1_out), count_fastq(r2_out)
    expected = READ_BUDGETS[sample]
    if n1 != n2:
        raise RuntimeError(f"{sample}: generated R1/R2 mismatch {n1} vs {n2}")
    if n1 != expected:
        print(f"WARNING {sample}: ISS generated {n1:,} pairs, readcount budget was {expected:,}")
    print(f"[{sample}] generated {n1:,} PE pairs")

for sample in SAMPLES:
    run_iss_generate(sample)


## 12. Final validation

In [ ]:
final_rows = []
for sample in SAMPLES:
    ext = ".fastq.gz" if COMPRESS else ".fastq"
    r1 = FASTQ_DIR / f"{sample}_R1{ext}"
    r2 = FASTQ_DIR / f"{sample}_R2{ext}"
    n1, n2 = count_fastq(r1), count_fastq(r2)
    final_rows.append({
        "sample": sample,
        "expected_pairs": READ_BUDGETS[sample],
        "R1_reads": n1,
        "R2_reads": n2,
        "paired_equal": n1 == n2,
        "budget_equal": n1 == READ_BUDGETS[sample],
    })

out = QC_DIR / "final_qc.tsv"
with open(out, "w", newline="") as h:
    w = csv.DictWriter(h, fieldnames=final_rows[0].keys(), delimiter="\t")
    w.writeheader(); w.writerows(final_rows)
for r in final_rows:
    print(r)
print("wrote", out)

## Interpretation / caveats

- **Dedup:** exact-sequence dedup is deterministic and does not merge near-identical BCRs, so it does not erase SHM variants by clustering.
- **No UMI:** multiplicity of an identical sequence cannot be decomposed into PCR copies versus independent starting molecules. Therefore `one_per_unique` is the default; use `observed_multiplicity` only as a sensitivity analysis, not as known truth.
- **PCR#1 (pre-fragmentation):** branching model represents stochastic amplification and jackpotting on whole templates. Does not model polymerase-introduced sequence mutations, chimeras, or plateau/saturation chemistry.
- **Fragmentation:** `FRAGMENTS_PER_TEMPLATE=5` candidate windows per template is a simplification of physical fragmentation (which tiles a molecule into many overlapping fragments); every candidate fragment inherits the full `pcr_copies` of its parent template as its PCR#2 starting abundance, rather than each amplified molecule being fragmented independently. Known ceiling: coarse approximation of true per-molecule tiling depth; revisit if downstream stitching validation needs finer position coverage.
- **PCR#2 (library-prep, post-fragmentation):** same branching model applied independently per fragment, fewer cycles (`LIBRARY_PCR_CYCLES=12`) than PCR#1. Captures fragment-to-fragment amplification bias within the same template, but still no polymerase error/chimera modeling.
- **Error model:** the ISS KDE model is learned from real raw reads mapped to a deduplicated merged pseudo-reference. This is empirical but not independent ground truth; inspect `samtools flagstat` and `iss_model.log` before trusting it.
- **Sequencing:** zero-read templates are allowed. The exact sample-level read budget is preserved by multinomial allocation before ISS generation.
- **Reproducibility:** sample-specific deterministic seeds are used for PCR and read allocation.